___________________________________
***Extension of Evaluation:***
In this section we will dive deeper into evaluation of the recommender system

In [ ]:
from tqdm import tqdm
import numpy as np


# Comprehensive evaluation: Extended metrics at multiple K values

def eval_at_k_extended(engine, test_dict, k=10, show_progress=True, sample_size=None):
    """Extended evaluation with precision, recall, F1, and NDCG metrics"""
    users = list(test_dict.keys())

    if sample_size is not None and sample_size < len(users):
        np.random.seed(42)
        users = np.random.choice(users, size=sample_size, replace=False).tolist()

    hits = 0
    recalls = []
    precisions = []
    f1_scores = []
    dcg = 0.0
    idcg = 0.0

    iterator = tqdm(users, desc=f"Evaluating @K={k}", ncols=100, disable=not show_progress)

    for u in iterator:
        rel_items = test_dict[u]
        if not rel_items:
            continue
        recs = engine.recommend(user=u, n=k, filter_seen_items=True)
        items = recs["item"].astype(str).tolist() if recs is not None and not recs.empty else []

        if any(it in rel_items for it in items):
            hits += 1

        inter = [1 if it in rel_items else 0 for it in items]
        num_relevant_recommended = sum(inter)

        # Recall: relevant recommended / total relevant
        user_recall = 0.0
        if len(rel_items) > 0:
            user_recall = num_relevant_recommended / len(rel_items)
            recalls.append(user_recall)

        # Precision: relevant recommended / total recommended
        user_precision = 0.0
        if len(items) > 0:
            user_precision = num_relevant_recommended / len(items)
            precisions.append(user_precision)

        # F1 Score: harmonic mean of precision and recall
        if user_precision + user_recall > 0:
            user_f1 = 2 * (user_precision * user_recall) / (user_precision + user_recall)
            f1_scores.append(user_f1)

        # NDCG calculation
        for rank, rel in enumerate(inter, start=1):
            if rel:
                dcg += 1.0 / np.log2(rank + 1)
        r = min(len(rel_items), k)
        idcg += sum(1.0 / np.log2(rk + 1) for rk in range(1, r + 1))

    hitrate = hits / len(users) if users else 0.0
    recall = float(np.mean(recalls)) if recalls else 0.0
    precision = float(np.mean(precisions)) if precisions else 0.0
    f1 = float(np.mean(f1_scores)) if f1_scores else 0.0
    ndcg = (dcg / idcg) if idcg > 1e-9 else 0.0

    return {
        "HitRate@K": round(hitrate, 4),
        "Precision@K": round(precision, 4),
        "Recall@K": round(recall, 4),
        "F1@K": round(f1, 4),
        "NDCG@K": round(ndcg, 4)
    }


# Evaluate enhanced model at multiple K values with extended metrics
print("\n" + "=" * 70)
print("📊 ENHANCED MODEL - COMPREHENSIVE EVALUATION")
print("=" * 70)

k_values = [5, 10, 20, 50]
all_results = {}

for k in k_values:
    print(f"\n⏳ Evaluating @ K={k}...")
    metrics = eval_at_k_extended(enhanced_engine, test_dict, k=k, sample_size=5000, show_progress=True)
    all_results[k] = metrics

    print(f"✓ K={k:2d} → HitRate: {metrics['HitRate@K']:.4f} | Precision: {metrics['Precision@K']:.4f} | Recall: {metrics['Recall@K']:.4f} | F1: {metrics['F1@K']:.4f} | NDCG: {metrics['NDCG@K']:.4f}")

# Display results in a nice table format
print("\n" + "=" * 70)
print("📈 PERFORMANCE SUMMARY TABLE")
print("=" * 70)
print(f"{'Metric':<15} | K=5    | K=10   | K=20   | K=50")
print("-" * 70)

for metric in ['HitRate@K', 'Precision@K', 'Recall@K', 'F1@K', 'NDCG@K']:
    values = [all_results[k][metric] for k in k_values]
    print(f"{metric:<15} | {values[0]:.4f} | {values[1]:.4f} | {values[2]:.4f} | {values[3]:.4f}")

# Visualize trends for each metric
print("\n" + "=" * 70)
print("📊 PERFORMANCE TRENDS BY METRIC")
print("=" * 70)

for metric in ['HitRate@K', 'Precision@K', 'Recall@K', 'F1@K', 'NDCG@K']:
    print(f"\n{metric}:")
    for k in k_values:
        value = all_results[k][metric]
        bar_length = int(value * 40)
        bar = "█" * bar_length + "░" * (40 - bar_length)
        print(f"  K={k:2d} : {value:.4f} │{bar}│")

print("\n" + "=" * 70)
print("💡 Insights:")
print("   • HitRate: % of users who got ≥1 relevant recommendation")
print("   • Precision: % of recommendations that were relevant")
print("   • Recall: % of user's test items that were recommended")
print("   • F1: Harmonic mean of Precision and Recall (balanced measure)")
print("   • NDCG: Ranking quality (higher = relevant items ranked higher)")
print("=" * 70)

all_results

In [ ]:
# Evaluation of how diverse the recommendations are
from typing import Counter

def analyze_coverage(engine, test_dict, k=10, sample_size=5000):
    """Analyze recommendation diversity and coverage"""
    users = list(test_dict.keys())

    if sample_size is not None and sample_size < len(users):
        np.random.seed(42)
        users = np.random.choice(users, size=sample_size, replace=False).tolist()

    all_recommended = set()
    recommendation_counts = Counter()

    for u in tqdm(users, desc="Coverage analysis", ncols=100):
        recs = engine.recommend(user=u, n=k, filter_seen_items=True)
        if recs is not None and not recs.empty:
            items = recs["item"].astype(str).tolist()
            all_recommended.update(items)
            recommendation_counts.update(items)

    total_items = len(enhanced_cb.items_)
    coverage = len(all_recommended) / total_items

    # Gini coefficient for concentration
    counts = sorted(recommendation_counts.values())
    n = len(counts)
    gini = 0
    if n > 0:
        cumsum = np.cumsum(counts)
        gini = (2 * np.sum((np.arange(1, n+1)) * counts)) / (n * cumsum[-1]) - (n + 1) / n

    print("\n" + "=" * 70)
    print("📊 RECOMMENDATION DIVERSITY ANALYSIS")
    print("=" * 70)
    print(f"Total unique items in catalog    : {total_items:,}")
    print(f"Items recommended at least once  : {len(all_recommended):,}")
    print(f"Coverage (% of catalog)          : {coverage*100:.2f}%")
    print(f"Gini coefficient (concentration) : {gini:.4f} (0=equal, 1=concentrated)")
    print(f"\nMost frequently recommended items:")
    for item, count in recommendation_counts.most_common(10):
        print(f"  {item[:60]:60s} → {count:4d} times ({count/len(users)*100:.1f}%)")
    print("=" * 70)

# Run coverage analysis
analyze_coverage(enhanced_engine, test_dict, k=10, sample_size=5000)

___________________
***Hyperparameter Tuning***
In order to find the best set of parameters, we will perform Bayesian Hyperparameter Tuning on the params of the model

In [ ]:
# Optuna Hyperparameter Optimization for Enhanced Content-Based Model

import time
import optuna
from optuna.samplers import TPESampler

# Track all trial results for analysis
all_trials_results = []

def objective(trial):
    """
    Objective function for Optuna optimization
    Returns F1 score (Optuna maximizes by default)
    """
    start_time = time.time()

    try:
        # Suggest hyperparameters
        min_df = trial.suggest_int('min_df', 2, 10)
        max_df = trial.suggest_float('max_df', 0.5, 0.9)
        max_features = trial.suggest_int('max_features', 50000, 150000, step=10000)
        ngram_range = trial.suggest_categorical('ngram_range', [(1, 1), (1, 2), (1, 3)])
        use_stemming = trial.suggest_categorical('use_stemming', [True, False])
        like_threshold = trial.suggest_float('like_threshold', 3.0, 4.5)

        print(f"\n{'='*70}")
        print(f"🔍 Trial {trial.number + 1}")
        print(f"   min_df={min_df}, max_df={max_df:.2f}, max_features={max_features}")
        print(f"   ngram_range={ngram_range}, use_stemming={use_stemming}")
        print(f"   like_threshold={like_threshold:.2f}")

        # Create model with suggested hyperparameters
        model = EnhancedContentBasedAlgo(
            min_df=min_df,
            max_df=max_df,
            max_features=max_features,
            ngram_range=ngram_range,
            use_stemming=use_stemming,
            use_props=USE_PROPS,
            prop_weight=1.0,
            like_threshold=like_threshold
        )

        # Use same enhanced popularity fallback
        fallback = EnhancedPopularityAlgo(
            mode="bayesian",
            min_count=5,
            global_mean=train_df["rating"].mean()
        )

        # Train model
        engine = RecEngine(algo=model, fallback=fallback)
        engine.fit(train_df)

        # Evaluate on validation set (use smaller sample for speed)
        metrics = eval_at_k_extended(
            engine,
            test_dict,
            k=10,
            sample_size=2000,  # Smaller sample for faster tuning
            show_progress=False
        )

        # Use F1 score as primary metric
        f1_score = metrics['F1@K']

        elapsed = time.time() - start_time

        print(f"📊 Results: F1={f1_score:.4f} | Precision={metrics['Precision@K']:.4f} | Recall={metrics['Recall@K']:.4f} | NDCG={metrics['NDCG@K']:.4f}")
        print(f"⏱️  Time: {elapsed:.1f}s")

        # Store trial results
        all_trials_results.append({
            'trial': trial.number,
            'params': trial.params.copy(),
            'metrics': metrics,
            'f1_score': f1_score,
            'time': elapsed
        })

        # Log additional metrics to Optuna
        trial.set_user_attr('precision', metrics['Precision@K'])
        trial.set_user_attr('recall', metrics['Recall@K'])
        trial.set_user_attr('ndcg', metrics['NDCG@K'])
        trial.set_user_attr('hitrate', metrics['HitRate@K'])
        trial.set_user_attr('time', elapsed)

        return f1_score

    except Exception as e:
        print(f"❌ Error in trial {trial.number}: {e}")
        return 0.0  # Return worst score on error


# Create Optuna study
print("\n" + "=" * 70)
print("🎯 OPTUNA HYPERPARAMETER OPTIMIZATION")
print("=" * 70)
print("Optimization metric: F1@K=10 (maximizing)")
print("Validation sample: 2,000 users per trial")
print("Sampler: TPE (Tree-structured Parzen Estimator)")
print("=" * 70)

study = optuna.create_study(
    direction='maximize',  # Maximize F1 score
    sampler=TPESampler(seed=42),
    study_name='content_based_optimization'
)

# Run optimization
n_trials = 20  # Number of trials (increase for better results)

study.optimize(
    objective,
    n_trials=n_trials,
    show_progress_bar=True,
    n_jobs=1  # Set to -1 for parallel trials (if you have multiple cores)
)

# Print results
print("\n" + "=" * 70)
print("🏆 OPTIMIZATION COMPLETE!")
print("=" * 70)
print(f"Number of finished trials: {len(study.trials)}")
print(f"Best trial: #{study.best_trial.number}")
print(f"Best F1 Score: {study.best_value:.4f}")
print(f"\n✨ Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"   {key:15s} = {value}")
print("=" * 70)

# Show additional stats
print(f"\n📈 Best Trial Details:")
best_trial = study.best_trial
print(f"   Precision@K  : {best_trial.user_attrs['precision']:.4f}")
print(f"   Recall@K     : {best_trial.user_attrs['recall']:.4f}")
print(f"   F1@K         : {study.best_value:.4f}")
print(f"   NDCG@K       : {best_trial.user_attrs['ndcg']:.4f}")
print(f"   HitRate@K    : {best_trial.user_attrs['hitrate']:.4f}")
print(f"   Training Time: {best_trial.user_attrs['time']:.1f}s")
print("=" * 70)

In [ ]:
# Visualize optimization results with Optuna's built-in plots

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Optuna Hyperparameter Optimization Results', fontsize=16, fontweight='bold')

# Plot 1: Optimization history
ax1 = axes[0, 0]
trials = study.trials
trial_numbers = [t.number for t in trials]
trial_values = [t.value for t in trials]
ax1.plot(trial_numbers, trial_values, 'o-', linewidth=2, markersize=6, alpha=0.7)
ax1.axhline(y=study.best_value, color='r', linestyle='--', linewidth=2, label=f'Best: {study.best_value:.4f}')
ax1.set_xlabel('Trial', fontsize=11)
ax1.set_ylabel('F1 Score', fontsize=11)
ax1.set_title('Optimization History', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot 2: All metrics progress
ax2 = axes[0, 1]
for metric in ['Precision@K', 'Recall@K', 'F1@K', 'NDCG@K']:
    values = [r['metrics'][metric] for r in all_trials_results]
    ax2.plot(range(len(values)), values, 'o-', label=metric, linewidth=2, markersize=4)
ax2.set_xlabel('Trial', fontsize=11)
ax2.set_ylabel('Metric Value', fontsize=11)
ax2.set_title('All Metrics Progress', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

# Plot 3: Parameter importance (if enough trials)
ax3 = axes[1, 0]
if len(study.trials) >= 10:
    try:
        importances = optuna.importance.get_param_importances(study)
        params = list(importances.keys())
        values = list(importances.values())
        colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(params)))
        bars = ax3.barh(params, values, color=colors, alpha=0.8)
        ax3.set_xlabel('Importance', fontsize=11)
        ax3.set_title('Hyperparameter Importance', fontsize=12, fontweight='bold')
        ax3.grid(alpha=0.3, axis='x')
        for bar, val in zip(bars, values):
            ax3.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                    f'{val:.3f}', va='center', fontsize=9)
    except Exception as e:
        ax3.text(0.5, 0.5, f'Need more trials\nfor importance\n({len(study.trials)} trials)',
                ha='center', va='center', transform=ax3.transAxes, fontsize=11)
        ax3.axis('off')
else:
    ax3.text(0.5, 0.5, 'Need ≥10 trials\nfor importance',
            ha='center', va='center', transform=ax3.transAxes, fontsize=11)
    ax3.axis('off')

# Plot 4: Top 5 trials comparison
ax4 = axes[1, 1]
top_5_trials = sorted(study.trials, key=lambda t: t.value, reverse=True)[:5]
trial_nums = [t.number for t in top_5_trials]
scores = [t.value for t in top_5_trials]
colors = ['gold', 'silver', '#CD7F32', 'lightblue', 'lightgreen']
bars = ax4.barh(range(len(trial_nums)), scores, color=colors, alpha=0.8)
ax4.set_yticks(range(len(trial_nums)))
ax4.set_yticklabels([f'Trial {n}' for n in trial_nums])
ax4.set_xlabel('F1 Score', fontsize=11)
ax4.set_title('Top 5 Trials', fontsize=12, fontweight='bold')
ax4.grid(alpha=0.3, axis='x')
for i, (bar, score) in enumerate(zip(bars, scores)):
    ax4.text(score + 0.0005, bar.get_y() + bar.get_height()/2,
             f'{score:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\n📊 Optimization Statistics:")
print(f"   Total trials      : {len(study.trials)}")
print(f"   Completed trials  : {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}")
print(f"   Failed trials     : {len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])}")
print(f"   Total time        : {sum(r['time'] for r in all_trials_results):.1f}s")
print(f"   Avg time/trial    : {sum(r['time'] for r in all_trials_results)/len(all_trials_results):.1f}s")
first_f1 = all_trials_results[0]['f1_score']
if first_f1 > 0:
    improvement = ((study.best_value - first_f1) / first_f1 * 100)
    print(f"   Improvement       : {improvement:+.1f}% over first trial")

In [ ]:
# Train final model with optimized hyperparameters and full evaluation

print("\n" + "=" * 70)
print("🎯 TRAINING FINAL MODEL WITH OPTIMIZED HYPERPARAMETERS")
print("=" * 70)

optimized_model = EnhancedContentBasedAlgo(
    min_df=study.best_params['min_df'],
    max_df=study.best_params['max_df'],
    max_features=study.best_params['max_features'],
    ngram_range=study.best_params['ngram_range'],
    use_stemming=study.best_params['use_stemming'],
    use_props=USE_PROPS,
    prop_weight=1.0,
    like_threshold=study.best_params['like_threshold']
)

optimized_fallback = EnhancedPopularityAlgo(
    mode="bayesian",
    min_count=5,
    global_mean=train_df["rating"].mean()
)

optimized_engine = RecEngine(algo=optimized_model, fallback=optimized_fallback)
optimized_engine.fit(train_df)

# Evaluate on full validation set
print("\n⏳ Evaluating optimized model on full validation set (5000 users)...")
optimized_metrics = eval_at_k_extended(
    optimized_engine,
    test_dict,
    k=10,
    sample_size=5000,
    show_progress=True
)

# Compare with baseline enhanced model
print("\n" + "=" * 70)
print("📊 FINAL COMPARISON: BASELINE ENHANCED vs OPTUNA-OPTIMIZED")
print("=" * 70)

print(f"\n{'Metric':<15} | Baseline | Optimized | Improvement")
print("-" * 70)
for metric in ['HitRate@K', 'Precision@K', 'Recall@K', 'F1@K', 'NDCG@K']:
    baseline_val = enhanced_metrics[metric]
    optimized_val = optimized_metrics[metric]
    if baseline_val > 0:
        improvement = ((optimized_val - baseline_val) / baseline_val * 100)
        print(f"{metric:<15} | {baseline_val:.4f}   | {optimized_val:.4f}    | {improvement:+.2f}%")
    else:
        print(f"{metric:<15} | {baseline_val:.4f}   | {optimized_val:.4f}    | N/A")

print("\n✨ Optimized Hyperparameters:")
for param, value in study.best_params.items():
    print(f"   {param:15s} = {value}")
print("=" * 70)

optimized_metrics